# TxnAgentBench Colab runner
Use an A100 runtime. Every experiment writes to persistent storage and can resume after disconnection.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
ARTIFACT_ROOT = '/content/drive/MyDrive/txnagentbench-artifacts'


In [ ]:
import os
REPO_URL = 'https://github.com/vnigoated/programming_.git'
BRANCH = 'master'
if not os.path.exists('/content/txnagentbench/.git'):
    !git clone --branch $BRANCH $REPO_URL /content/txnagentbench
%cd /content/txnagentbench
!git fetch origin $BRANCH
!git checkout $BRANCH
!git pull --ff-only origin $BRANCH


In [ ]:
!bash scripts/bootstrap_colab.sh /content/txnagentbench
!nvidia-smi
!pytest -q


In [ ]:
SMOKE_OUT = f'{ARTIFACT_ROOT}/smoke'
!txnagentbench run --config configs/smoke.json --output $SMOKE_OUT --allow-scripted
!txnagentbench analyze --output $SMOKE_OUT
!cat $SMOKE_OUT/analysis/report.md


## Real model smoke test
Copy the pilot template, replace the model ID and immutable revision, then run only 20 executions. Do not start the primary matrix from this notebook until the runbook gates pass.

In [ ]:
PILOT_CONFIG = 'configs/pilot_local.json'  # create from pilot_template.json
MODEL_SMOKE_OUT = f'{ARTIFACT_ROOT}/model-smoke'
# !txnagentbench run --config $PILOT_CONFIG --output $MODEL_SMOKE_OUT --max-runs 20
# !txnagentbench analyze --output $MODEL_SMOKE_OUT


## Resume rule
After any Colab interruption, rerun setup and then repeat the same experiment command. Completed run IDs are skipped automatically. Never delete the artifact directory to fix a failed runtime.

In [6]:
from pathlib import Path
import subprocess

REMOTE = "https://github.com/vnigoated/programming_.git"
REPO = Path("/content/txnagentbench")
ARTIFACT_ROOT = Path("/content/drive/MyDrive/txnagentbench-artifacts")

assert ARTIFACT_ROOT.is_dir(), "Mount Google Drive first."

if not (REPO / ".git").is_dir():
    if REPO.exists():
        REPO = Path("/content/txnagentbench-fresh")
    subprocess.run(
        ["git", "clone", REMOTE, str(REPO)],
        check=True,
    )
else:
    subprocess.run(
        ["git", "-C", str(REPO), "pull", "--ff-only", "origin", "master"],
        check=True,
    )

print("Repository:", REPO)
print(
    "Commit:",
    subprocess.check_output(
        ["git", "-C", str(REPO), "rev-parse", "HEAD"],
        text=True,
    ).strip(),
)

process = subprocess.Popen(
    [
        "bash",
        str(REPO / "scripts/run_expanded_pilot_qwen7b.sh"),
        str(REPO),
        str(ARTIFACT_ROOT),
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in process.stdout:
    print(line, end="")

return_code = process.wait()
print("\nFinal return code:", return_code)

if return_code:
    raise RuntimeError(f"Pilot wrapper failed with code {return_code}")

Repository: /content/txnagentbench
Commit: f1993d8ec7d8b0d0057d6654b0348bf4d0144ac5
GPU: NVIDIA A100-SXM4-40GB
GPU memory GiB: 39.49
Obtaining file:///content/txnagentbench
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Building editable for txnagentbench (pyproject.toml): started
  Building editable for txnagentbench (pyproject.toml): finished with status 'done'
  Created wheel for txnagentbench: filename=txnagentbench-0.1.0-0.editable-py3-none-any.whl size=1799 sha256=d70dcb18260dbbb3d649e085092f165156cf8125f6177ad4a43c06a5251ee945
  Stored 

RuntimeError: Pilot wrapper failed with code 1

In [7]:
import subprocess

REPO = "/content/txnagentbench"
ARTIFACT_ROOT = "/content/drive/MyDrive/txnagentbench-artifacts"

subprocess.run(
    ["git", "-C", REPO, "pull", "--ff-only", "origin", "master"],
    check=True,
)

print(
    "Commit:",
    subprocess.check_output(
        ["git", "-C", REPO, "rev-parse", "HEAD"],
        text=True,
    ).strip(),
)

process = subprocess.Popen(
    [
        "bash",
        f"{REPO}/scripts/run_expanded_pilot_qwen7b.sh",
        REPO,
        ARTIFACT_ROOT,
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in process.stdout:
    print(line, end="")

return_code = process.wait()
print("\nFinal return code:", return_code)

if return_code:
    raise RuntimeError(f"Pilot wrapper failed with code {return_code}")

Commit: 7b0c9e3c82e87ea12a6ffcb04d9fd221645e5427
GPU: NVIDIA A100-SXM4-40GB
GPU memory GiB: 39.49
Obtaining file:///content/txnagentbench
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Building editable for txnagentbench (pyproject.toml): started
  Building editable for txnagentbench (pyproject.toml): finished with status 'done'
  Created wheel for txnagentbench: filename=txnagentbench-0.1.0-0.editable-py3-none-any.whl size=1799 sha256=98d4d558c5abfecd0c4c36f74553ec08258d5be0b1dc9f393b2250cce6ffcfcc
  Stored in directory: /tmp/pip-ephem-wheel-

RuntimeError: Pilot wrapper failed with code 1

In [2]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


In [9]:
import subprocess

REPO = "/content/txnagentbench"
ARTIFACT_ROOT = "/content/drive/MyDrive/txnagentbench-artifacts"

subprocess.run(
    ["git", "-C", REPO, "pull", "--ff-only", "origin", "master"],
    check=True,
)

subprocess.run(
    [
        "bash",
        f"{REPO}/scripts/run_expanded_pilot_qwen7b.sh",
        REPO,
        ARTIFACT_ROOT,
    ],
    check=True,
)

CompletedProcess(args=['bash', '/content/txnagentbench/scripts/run_expanded_pilot_qwen7b.sh', '/content/txnagentbench', '/content/drive/MyDrive/txnagentbench-artifacts'], returncode=0)

In [3]:
from google.colab import drive
drive.mount("/content/drive")

import json
import subprocess
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

REPO = Path("/content/txnagentbench")
ROOT = Path(
    "/content/drive/MyDrive/txnagentbench-artifacts/"
    "expanded-pilot-qwen7b-v2"
)

# Restore repository in a fresh runtime.
if not (REPO / ".git").exists():
    subprocess.run(
        [
            "git",
            "clone",
            "https://github.com/vnigoated/programming_.git",
            str(REPO),
        ],
        check=True,
    )
else:
    subprocess.run(
        ["git", "-C", str(REPO), "pull", "--ff-only", "origin", "master"],
        check=True,
    )

# CPU-only installation.
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-e",
        str(REPO),
    ],
    check=True,
)

assert ROOT.exists(), f"Artifact directory not found: {ROOT}"

blocks = ["core", "dependency", "recovery"]
total_runs = 0

for name in blocks:
    block = ROOT / name

    verification = subprocess.run(
        [
            sys.executable,
            "-m",
            "txnagentbench.cli",
            "verify-manifest",
            "--output",
            str(block),
        ],
        check=True,
        capture_output=True,
        text=True,
    )
    print(verification.stdout.strip())

    manifest = json.loads(
        (block / "artifact_manifest.json").read_text()
    )
    policy_metrics = pd.read_csv(
        block / "analysis" / "policy_metrics.csv"
    )
    paired = pd.read_csv(
        block / "analysis" / "paired_comparisons.csv"
    )

    completed = manifest["validation"]["completed_runs"]
    total_runs += completed

    print("\n" + "=" * 80)
    print(f"BLOCK: {name}")
    print(f"Completed runs: {completed}")
    print(f"Commit: {manifest['code_commit']}")
    print(f"Files verified: {len(manifest['files'])}")

    print("\nPolicy metrics:")
    display(policy_metrics)

    print("\nPaired comparisons:")
    display(paired)

assert total_runs == 3264, f"Expected 3264 runs, found {total_runs}"

print("\n" + "=" * 80)
print("PASS: all expanded-pilot blocks are valid and immutable")
print("Total completed runs:", total_runs)

Mounted at /content/drive


KeyboardInterrupt: 

In [4]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import display

ROOT = Path(
    "/content/drive/MyDrive/txnagentbench-artifacts/"
    "expanded-pilot-qwen7b-v2"
)

expected_runs = {
    "core": 1536,
    "dependency": 192,
    "recovery": 1536,
}

total_runs = 0

for name, expected in expected_runs.items():
    block = ROOT / name
    manifest_path = block / "artifact_manifest.json"
    metrics_path = block / "analysis" / "policy_metrics.csv"
    paired_path = block / "analysis" / "paired_comparisons.csv"

    assert manifest_path.exists(), f"Missing manifest: {manifest_path}"
    assert metrics_path.exists(), f"Missing metrics: {metrics_path}"
    assert paired_path.exists(), f"Missing paired results: {paired_path}"

    manifest = json.loads(manifest_path.read_text())
    completed = manifest["validation"]["completed_runs"]

    assert completed == expected, (
        f"{name}: expected {expected}, found {completed}"
    )

    total_runs += completed

    print("\n" + "=" * 80)
    print(f"BLOCK: {name}")
    print(f"Completed runs: {completed}")
    print(f"Code commit: {manifest['code_commit']}")
    print(f"Files in manifest: {len(manifest['files'])}")

    print("\nPolicy metrics:")
    display(pd.read_csv(metrics_path))

    print("\nPaired comparisons:")
    display(pd.read_csv(paired_path))

assert total_runs == 3264

print("\n" + "=" * 80)
print("PASS: all three frozen blocks are present")
print("Total completed runs:", total_runs)


BLOCK: core
Completed runs: 1536
Code commit: f1993d8ec7d8b0d0057d6654b0348bf4d0144ac5
Files in manifest: 3082

Policy metrics:


,policy,runs,task_success_rate,severe_violation_rate,lost_update_rate,side_effect_violation_rate,dependency_violation_rate,invalid_action_rate,unsafe_proposal_rate,recovery_attempts,recovery_successes,recovery_rate,recovery_violation_rate,mean_conflicts,median_duration_seconds,mean_total_tokens
0,mtpo_reproduction,256,1.0000,0.0000,0.0000,0.000000,0.0,0.0,0.0,204,204,1.0,0.000,0.750000,1.230650,987.250000
1,naive,256,0.4375,0.5625,0.5625,0.492188,0.0,0.0,0.0,288,0,0.0,0.375,0.000000,1.040954,790.031250
2,occ,256,0.6250,0.3750,0.3750,0.328125,0.0,0.0,0.0,204,0,0.0,0.375,0.750000,1.032982,790.031250
3,postgres_serializable,256,0.6250,0.3750,0.3750,0.328125,0.0,0.0,0.0,204,0,0.0,0.375,1.472656,1.270198,1082.324219
4,serial,256,0.6250,0.3750,0.3750,0.328125,0.0,0.0,0.0,204,0,0.0,0.375,0.000000,1.036498,791.062500
5,txnguard,256,1.0000,0.0000,0.0000,0.000000,0.0,0.0,0.0,204,204,1.0,0.000,0.750000,1.036773,790.031250



Paired comparisons:


,baseline,pairs,risk_difference,ci95_low,ci95_high,baseline_only_failure,txnguard_only_failure,mcnemar_exact_p
0,naive,256,-0.5625,-0.625000,-0.5000,144,0,8.968310e-44
1,occ,256,-0.3750,-0.433594,-0.3125,96,0,2.524355e-29
2,serial,256,-0.3750,-0.433594,-0.3125,96,0,2.524355e-29



BLOCK: dependency
Completed runs: 192
Code commit: f1993d8ec7d8b0d0057d6654b0348bf4d0144ac5
Files in manifest: 394

Policy metrics:


...


BLOCK: recovery
Completed runs: 1536
Code commit: f1993d8ec7d8b0d0057d6654b0348bf4d0144ac5
Files in manifest: 3082

Policy metrics:


,policy,runs,task_success_rate,severe_violation_rate,lost_update_rate,side_effect_violation_rate,dependency_violation_rate,invalid_action_rate,unsafe_proposal_rate,recovery_attempts,recovery_successes,recovery_rate,recovery_violation_rate,mean_conflicts,median_duration_seconds,mean_total_tokens
0,mtpo_reproduction,256,1.0,0.0,0.000,0.000,0.0,0.0,0.0,640,640,1.0,0.0,1.000000,1.207004,1052.250000
1,naive,256,0.0,1.0,0.625,0.625,0.0,0.0,0.0,768,0,0.0,1.0,0.000000,1.189280,789.375000
2,occ,256,0.0,1.0,0.500,0.500,0.0,0.0,0.0,576,0,0.0,1.0,0.750000,1.008612,789.375000
3,postgres_serializable,256,0.0,1.0,0.500,0.500,0.0,0.0,0.0,576,0,0.0,1.0,1.175781,1.243920,1011.632812
4,serial,256,0.0,1.0,0.500,0.500,0.0,0.0,0.0,576,0,0.0,1.0,0.000000,1.199651,790.125000
5,txnguard,256,1.0,0.0,0.000,0.000,0.0,0.0,0.0,576,576,1.0,0.0,1.000000,1.011902,789.375000



Paired comparisons:


,baseline,pairs,risk_difference,ci95_low,ci95_high,baseline_only_failure,txnguard_only_failure,mcnemar_exact_p
0,naive,256,-1.0,-1.0,-1.0,256,0,1.727234e-77
1,occ,256,-1.0,-1.0,-1.0,256,0,1.727234e-77
2,serial,256,-1.0,-1.0,-1.0,256,0,1.727234e-77



PASS: all three frozen blocks are present
Total completed runs: 3264


In [5]:
from pathlib import Path

import pandas as pd
from IPython.display import display

ROOT = Path(
    "/content/drive/MyDrive/txnagentbench-artifacts/"
    "expanded-pilot-qwen7b-v2"
)

blocks = ["core", "dependency", "recovery"]
tables = []

for block in blocks:
    path = ROOT / block / "analysis" / "fault_policy_metrics.csv"
    table = pd.read_csv(path)
    table.insert(0, "block", block)
    tables.append(table)

fault_results = pd.concat(tables, ignore_index=True)

columns = [
    "block",
    "fault_mode",
    "policy",
    "runs",
    "task_success_rate",
    "severe_violation_rate",
    "lost_update_rate",
    "side_effect_violation_rate",
    "recovery_attempts",
    "recovery_successes",
    "recovery_rate",
    "recovery_violation_rate",
]

fault_results = fault_results[columns].sort_values(
    ["block", "fault_mode", "policy"]
)

for (block, fault), group in fault_results.groupby(
    ["block", "fault_mode"],
    sort=False,
):
    print("\n" + "=" * 90)
    print(f"BLOCK: {block} | FAULT: {fault}")
    display(group.drop(columns=["block", "fault_mode"]).reset_index(drop=True))

print("\nFault modes found:")
print(fault_results.groupby("block")["fault_mode"].unique())

print("\nRuns by block and fault:")
display(
    fault_results.groupby(["block", "fault_mode"], as_index=False)["runs"]
    .sum()
)


BLOCK: core | FAULT: duplicate_effect


,policy,runs,task_success_rate,severe_violation_rate,lost_update_rate,side_effect_violation_rate,recovery_attempts,recovery_successes,recovery_rate,recovery_violation_rate
0,mtpo_reproduction,128,1.00,0.00,0.00,0.00000,204,204,1.0,0.00
1,naive,128,0.25,0.75,0.75,0.65625,288,0,0.0,0.75
2,occ,128,0.25,0.75,0.75,0.65625,204,0,0.0,0.75
3,postgres_serializable,128,0.25,0.75,0.75,0.65625,204,0,0.0,0.75
4,serial,128,0.25,0.75,0.75,0.65625,204,0,0.0,0.75
5,txnguard,128,1.00,0.00,0.00,0.00000,204,204,1.0,0.00



BLOCK: core | FAULT: none


,policy,runs,task_success_rate,severe_violation_rate,lost_update_rate,side_effect_violation_rate,recovery_attempts,recovery_successes,recovery_rate,recovery_violation_rate
0,mtpo_reproduction,128,1.000,0.000,0.000,0.000000,0,0,NaN,0.0
1,naive,128,0.625,0.375,0.375,0.328125,0,0,NaN,0.0
2,occ,128,1.000,0.000,0.000,0.000000,0,0,NaN,0.0
3,postgres_serializable,128,1.000,0.000,0.000,0.000000,0,0,NaN,0.0
4,serial,128,1.000,0.000,0.000,0.000000,0,0,NaN,0.0
5,txnguard,128,1.000,0.000,0.000,0.000000,0,0,NaN,0.0



BLOCK: dependency | FAULT: dependency_change


...

,policy,runs,task_success_rate,severe_violation_rate,lost_update_rate,side_effect_violation_rate,recovery_attempts,recovery_successes,recovery_rate,recovery_violation_rate
0,mtpo_reproduction,64,1.0,0.0,0.0,0.0,192,192,1.0,0.0
1,naive,64,0.0,1.0,0.0,0.0,192,0,0.0,1.0
2,occ,64,0.0,1.0,0.0,0.0,192,0,0.0,1.0
3,postgres_serializable,64,0.0,1.0,0.0,0.0,192,0,0.0,1.0
4,serial,64,0.0,1.0,0.0,0.0,192,0,0.0,1.0
5,txnguard,64,1.0,0.0,0.0,0.0,128,128,1.0,0.0



BLOCK: recovery | FAULT: delayed_ack


,policy,runs,task_success_rate,severe_violation_rate,lost_update_rate,side_effect_violation_rate,recovery_attempts,recovery_successes,recovery_rate,recovery_violation_rate
0,mtpo_reproduction,64,1.0,0.0,0.0,0.0,128,128,1.0,0.0
1,naive,64,0.0,1.0,1.0,1.0,192,0,0.0,1.0
2,occ,64,0.0,1.0,1.0,1.0,128,0,0.0,1.0
3,postgres_serializable,64,0.0,1.0,1.0,1.0,128,0,0.0,1.0
4,serial,64,0.0,1.0,1.0,1.0,128,0,0.0,1.0
5,txnguard,64,1.0,0.0,0.0,0.0,128,128,1.0,0.0



Fault modes found:
block
core                                   [duplicate_effect, none]
dependency                                  [dependency_change]
recovery      [compensation_failure, crash_after_commit_befo...
Name: fault_mode, dtype: object

Runs by block and fault:


,block,fault_mode,runs
0,core,duplicate_effect,768
1,core,none,768
2,dependency,dependency_change,192
3,recovery,compensation_failure,384
4,recovery,crash_after_commit_before_ack,384
5,recovery,crash_before_commit,384
6,recovery,delayed_ack,384


In [1]:
from google.colab import drive
drive.mount("/content/drive")

import subprocess
from pathlib import Path

REPO = Path("/content/txnagentbench")
ARTIFACT_ROOT = "/content/drive/MyDrive/txnagentbench-artifacts"

if not (REPO / ".git").exists():
    subprocess.run(
        [
            "git",
            "clone",
            "https://github.com/vnigoated/programming_.git",
            str(REPO),
        ],
        check=True,
    )
else:
    status = subprocess.check_output(
        ["git", "-C", str(REPO), "status", "--porcelain"],
        text=True,
    ).strip()
    assert not status, f"Repository has local changes:\n{status}"

    subprocess.run(
        ["git", "-C", str(REPO), "pull", "--ff-only", "origin", "master"],
        check=True,
    )

print(
    "Commit:",
    subprocess.check_output(
        ["git", "-C", str(REPO), "rev-parse", "HEAD"],
        text=True,
    ).strip(),
)

subprocess.run(
    [
        "bash",
        str(REPO / "scripts/run_mistral_model_smoke.sh"),
        str(REPO),
        ARTIFACT_ROOT,
    ],
    check=True,
)

Mounted at /content/drive
Commit: 4b5bd231aeba684fbbca1f80592638217979313e


CompletedProcess(args=['bash', '/content/txnagentbench/scripts/run_mistral_model_smoke.sh', '/content/txnagentbench', '/content/drive/MyDrive/txnagentbench-artifacts'], returncode=0)

In [2]:
import subprocess

REPO = "/content/txnagentbench"
ARTIFACT_ROOT = "/content/drive/MyDrive/txnagentbench-artifacts"

subprocess.run(
    ["git", "-C", REPO, "pull", "--ff-only", "origin", "master"],
    check=True,
)

print(
    "Commit:",
    subprocess.check_output(
        ["git", "-C", REPO, "rev-parse", "HEAD"],
        text=True,
    ).strip(),
)

subprocess.run(
    [
        "bash",
        f"{REPO}/scripts/run_cross_model_ablation_smoke.sh",
        REPO,
        ARTIFACT_ROOT,
    ],
    check=True,
)

Commit: e15a5297774ff5f6782bc30ad1fdec0417e6277a


CompletedProcess(args=['bash', '/content/txnagentbench/scripts/run_cross_model_ablation_smoke.sh', '/content/txnagentbench', '/content/drive/MyDrive/txnagentbench-artifacts'], returncode=0)

In [1]:
from google.colab import drive
drive.mount("/content/drive")

import subprocess
from pathlib import Path

REPO = Path("/content/txnagentbench")
ARTIFACT_ROOT = "/content/drive/MyDrive/txnagentbench-artifacts"
FREEZE_TAG = "txnagentbench-primary-v1-freeze"
EXPECTED_COMMIT = "cd4d317c6f547bd4e3540ba696b2b93a37782ed5"

if not (REPO / ".git").exists():
    subprocess.run(
        [
            "git",
            "clone",
            "https://github.com/vnigoated/programming_.git",
            str(REPO),
        ],
        check=True,
    )

status = subprocess.check_output(
    ["git", "-C", str(REPO), "status", "--porcelain"],
    text=True,
).strip()
assert not status, f"Repository has local changes:\n{status}"

subprocess.run(
    ["git", "-C", str(REPO), "fetch", "origin", "--tags"],
    check=True,
)

subprocess.run(
    ["git", "-C", str(REPO), "checkout", "--detach", FREEZE_TAG],
    check=True,
)

commit = subprocess.check_output(
    ["git", "-C", str(REPO), "rev-parse", "HEAD"],
    text=True,
).strip()

assert commit == EXPECTED_COMMIT, (commit, EXPECTED_COMMIT)
print("Primary freeze commit:", commit)

subprocess.run(
    [
        "bash",
        str(REPO / "scripts/run_primary_v1_batch.sh"),
        "dependency",
        str(REPO),
        ARTIFACT_ROOT,
        "500",
    ],
    check=True,
)

Mounted at /content/drive
Primary freeze commit: cd4d317c6f547bd4e3540ba696b2b93a37782ed5


CompletedProcess(args=['bash', '/content/txnagentbench/scripts/run_primary_v1_batch.sh', 'dependency', '/content/txnagentbench', '/content/drive/MyDrive/txnagentbench-artifacts', '500'], returncode=0)

In [2]:
from google.colab import drive
drive.mount("/content/drive")

import subprocess
import sys
from pathlib import Path

REPO = Path("/content/txnagentbench")
ARTIFACT_ROOT = Path(
    "/content/drive/MyDrive/txnagentbench-artifacts"
)
PRIMARY_ROOT = ARTIFACT_ROOT / "txnagentbench-primary-v1"
ANALYSIS_ROOT = (
    ARTIFACT_ROOT / "txnagentbench-primary-v1-analysis"
)

FREEZE_TAG = "txnagentbench-primary-v1-freeze"
EXPECTED_COMMIT = "cd4d317c6f547bd4e3540ba696b2b93a37782ed5"

BLOCKS = [
    ("dependency", 1728),
    ("concurrency", 6912),
    ("recovery", 6400),
]


def run(command):
    print("\nRUNNING:", " ".join(map(str, command)), flush=True)
    subprocess.run(list(map(str, command)), check=True)


def block_progress(block, expected):
    output = PRIMARY_ROOT / block

    completed = sum(
        1 for _ in (output / "runs").glob("*/summary.json")
    )
    errors = sum(
        1 for _ in (output / "runs").glob("*/error.json")
    )
    frozen = (output / "artifact_manifest.json").exists()

    print(
        f"{block}: {completed}/{expected} | "
        f"pending errors={errors} | frozen={frozen}",
        flush=True,
    )

    return completed, errors, frozen


# ---------------------------------------------------------
# 1. Restore repository if this is a fresh Colab runtime
# ---------------------------------------------------------

if not (REPO / ".git").exists():
    run(
        [
            "git",
            "clone",
            "https://github.com/vnigoated/programming_.git",
            REPO,
        ]
    )

status = subprocess.check_output(
    ["git", "-C", str(REPO), "status", "--porcelain"],
    text=True,
).strip()

assert not status, (
    "Repository contains local changes. Do not continue:\n"
    + status
)

# ---------------------------------------------------------
# 2. Lock repository to the immutable primary tag
# ---------------------------------------------------------

run(["git", "-C", REPO, "fetch", "origin", "--tags"])
run(
    [
        "git",
        "-C",
        REPO,
        "checkout",
        "--detach",
        FREEZE_TAG,
    ]
)

commit = subprocess.check_output(
    ["git", "-C", str(REPO), "rev-parse", "HEAD"],
    text=True,
).strip()

assert commit == EXPECTED_COMMIT, (
    f"Wrong commit: {commit}\n"
    f"Expected: {EXPECTED_COMMIT}"
)

print("Primary freeze commit:", commit, flush=True)

# ---------------------------------------------------------
# 3. Run or resume every primary block
# ---------------------------------------------------------

for block, expected in BLOCKS:
    print("\n" + "=" * 80)
    print(f"PRIMARY BLOCK: {block}", flush=True)

    completed, errors, frozen = block_progress(
        block,
        expected,
    )

    if frozen:
        print(f"PASS: {block} already frozen; skipping.")
        continue

    # A large limit avoids unnecessary model reloads.
    # Individual runs are still saved atomically.
    run(
        [
            "bash",
            REPO / "scripts/run_primary_v1_batch.sh",
            block,
            REPO,
            ARTIFACT_ROOT,
            str(expected),
        ]
    )

    completed, errors, frozen = block_progress(
        block,
        expected,
    )

    assert completed == expected, (
        f"{block} remains incomplete: "
        f"{completed}/{expected}"
    )
    assert errors == 0, (
        f"{block} has {errors} pending errors"
    )
    assert frozen, f"{block} was not frozen"

# ---------------------------------------------------------
# 4. Confirm that all three primary blocks are complete
# ---------------------------------------------------------

all_frozen = True

print("\n" + "=" * 80)
print("FINAL PRIMARY CHECK")

for block, expected in BLOCKS:
    completed, errors, frozen = block_progress(
        block,
        expected,
    )

    all_frozen &= (
        completed == expected
        and errors == 0
        and frozen
    )

assert all_frozen, "Primary study is not yet complete."

print("PASS: all 15,040 primary runs are frozen.")

# ---------------------------------------------------------
# 5. Run the prespecified final statistical analysis
# ---------------------------------------------------------

run(
    [
        sys.executable,
        "-m",
        "txnagentbench.cli",
        "analyze-primary",
        "--input",
        PRIMARY_ROOT / "concurrency",
        "--input",
        PRIMARY_ROOT / "dependency",
        "--input",
        PRIMARY_ROOT / "recovery",
        "--plan",
        REPO / "configs/primary_v1_analysis_plan.json",
        "--output",
        ANALYSIS_ROOT,
    ]
)

assert (
    ANALYSIS_ROOT / "primary_results.json"
).exists()

print("\n" + "=" * 80)
print("PASS: PRIMARY STUDY AND ANALYSIS COMPLETE")
print("Runs: 15,040")
print("Artifacts:", PRIMARY_ROOT)
print("Analysis:", ANALYSIS_ROOT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

RUNNING: git -C /content/txnagentbench fetch origin --tags

RUNNING: git -C /content/txnagentbench checkout --detach txnagentbench-primary-v1-freeze
Primary freeze commit: cd4d317c6f547bd4e3540ba696b2b93a37782ed5

PRIMARY BLOCK: dependency
dependency: 500/1728 | pending errors=0 | frozen=False

RUNNING: bash /content/txnagentbench/scripts/run_primary_v1_batch.sh dependency /content/txnagentbench /content/drive/MyDrive/txnagentbench-artifacts 1728
dependency: 1728/1728 | pending errors=0 | frozen=True

PRIMARY BLOCK: concurrency
concurrency: 0/6912 | pending errors=0 | frozen=False

RUNNING: bash /content/txnagentbench/scripts/run_primary_v1_batch.sh concurrency /content/txnagentbench /content/drive/MyDrive/txnagentbench-artifacts 6912


KeyboardInterrupt: 